# Basic Decision Tree from Scratch - Classification (Median)

In this notebook, we are going to build a decision tree from scratch using median as a cutoff point. 

**Please note that in real decision tree we use Gini to compute the cutoff point for split. In this notebook, we delay that and only use median. The purpose of this notebook is to teach ourself how to build a tree and make prediction with a tree.**

Dataset: This is a makeup dataset that describe how much study hours a student put in and whether if they will either pass or fail their exam

| Study Hours  | Pass Exam |
| ----- | ----- |
| 1    | No    |
| 2    | No    |
| 3    | No    |
| 4    | No    |
| 5    | Yes   |
| 6    | Yes   |
| 7    | Yes   |
| 8    | Yes   |
| 9    | Yes   |
| 10    | Yes   |
| 11    | Yes   |

In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [3]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


We are going to create a tree by using median as a cutoff. The steps are as follows:

Build(node_data):

1. If the dataset is empty return None

2. If all labels are identical:
      create leaf and return prediction

3. If there is only one row left:
      create leaf and return prediction

4. Compute median of feature

5. Split rows into:
      left  = values <= median
      right = values > median

4. Create node storing median

5. Build(left) Recursively

6. Build(right) Recursively

In [4]:
def built_tree(df):

    if df.empty:
        return None

    # If all labels are identical, return that label
    if df['passed'].nunique() == 1:
        return int(df['passed'].iloc[0])

    # If there's only one row left, return its label
    if len(df) <= 1:
        return int(df['passed'].iloc[0])

    median_studied = df['studied'].median()
    print(f"Median studied: {median_studied}")

    # Split the dataset into two subsets based on the median value of 'studied'
    left_subset = df[df['studied'] <= median_studied]
    right_subset = df[df['studied'] > median_studied]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node
    clf = {'median_studied': float(median_studied)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

In [5]:
clf = built_tree(df)

Median studied: 6.5
Left subset:
   studied  passed
0        1       0
1        2       0
2        3       0
3        4       0
4        5       1
5        6       1

Right subset:
    studied  passed
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1

Median studied: 3.5
Left subset:
   studied  passed
0        1       0
1        2       0
2        3       0

Right subset:
   studied  passed
3        4       0
4        5       1
5        6       1

Median studied: 5.0
Left subset:
   studied  passed
3        4       0
4        5       1

Right subset:
   studied  passed
5        6       1

Median studied: 4.5
Left subset:
   studied  passed
3        4       0

Right subset:
   studied  passed
4        5       1



In [6]:
clf


{'median_studied': 6.5,
 'left': {'median_studied': 3.5,
  'left': 0,
  'right': {'median_studied': 5.0,
   'left': {'median_studied': 4.5, 'left': 0, 'right': 1},
   'right': 1}},
 'right': 1}

In [7]:
# print tree structure in json format
import json
print(json.dumps(clf, indent=8))


{
        "median_studied": 6.5,
        "left": {
                "median_studied": 3.5,
                "left": 0,
                "right": {
                        "median_studied": 5.0,
                        "left": {
                                "median_studied": 4.5,
                                "left": 0,
                                "right": 1
                        },
                        "right": 1
                }
        },
        "right": 1
}


In [8]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Median studied: {node['median_studied']}")
        print(f"{indent}├─ Left:")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right:")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent}Passed: {node}")


In [9]:
print_tree(clf)

    Median studied: 6.5
    ├─ Left:
    │   Median studied: 3.5
    │   ├─ Left:
    │   │   Passed: 0
    │   └─ Right:
    │       Median studied: 5.0
    │       ├─ Left:
    │       │   Median studied: 4.5
    │       │   ├─ Left:
    │       │   │   Passed: 0
    │       │   └─ Right:
    │       │       Passed: 1
    │       └─ Right:
    │           Passed: 1
    └─ Right:
        Passed: 1


In [10]:
# make prediction function
def predict(clf, x):
    node = clf  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['median_studied']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [11]:
prediction = predict(clf, 6)
print(f"Prediction for student who studied 5 hours: {prediction}")

Going left: 6 <= 6.5
Going right: 6 > 3.5
Going right: 6 > 5.0
Prediction for student who studied 5 hours: 1
